In [1]:
import pandas as pd
import json
from random import Random
import numpy as np
import os
from tqdm import tqdm

random_seed = 123
slice_idx = 0

In [ ]:
filtered_gameids_path = '../data/filtered_gameids.json'
label_studio_input_path = '../data/label_studio_input/label_studio_input.json'
out_dir = os.path.abspath(f'../data/sampled_annotations/{slice_idx}')

if not os.path.isdir(out_dir):
    print(f'create path {out_dir}')
    os.makedirs(out_dir)

In [3]:
with open(filtered_gameids_path, 'r') as f:
    filtered_gameids = json.load(f)['gameids']

In [4]:
input_data = pd.read_json(label_studio_input_path)
# unpack data and meta columns
input_data = input_data.assign(**input_data.data.apply(pd.Series))
input_data = input_data.assign(**input_data.meta.apply(pd.Series))

In [5]:
games_to_sample = 25
gameids_sample = Random(random_seed).sample(filtered_gameids, k=games_to_sample)

In [6]:
slices = []
idx_map = []

for idx, game_id in enumerate(gameids_sample):
    data_slice = input_data.loc[input_data.game_id == game_id].sort_values(by='round_num')[['data', 'meta']]
    
    # assert number of entries
    assert len(data_slice) == 60, len(data_slice)
    # assert sorting
    assert all(data_slice.meta.map(lambda x: x.get('round_num')).values == np.arange(1,61))
    
    slices.append(
        (idx, game_id, data_slice)
    )
    
    idx_map.append(
        {'slice_idx': slice_idx, 'game_idx': idx, 'game_id': game_id}
    )

In [ ]:
# number of rounds
sum([len(s) for (_, _, s) in slices])

1500

In [ ]:
print(f'save files to {out_dir}')

for idx, game_id, slice in tqdm(slices):
    
    slice_dict = slice.to_dict(orient='records')
    
    filename = f'{slice_idx}_{idx}.json'
    out_path = os.path.join(out_dir, filename)
    
    with open(out_path, 'w') as f:
        json.dump(slice_dict, f)
        
idx_map_path = os.path.join(out_dir, f'idx_map_{slice_idx}.json')
print(f'save idx map to {idx_map_path}')

with open(idx_map_path, 'w') as f:
    json.dump(idx_map, f)

In [8]:
pd.read_json(idx_map_path)

,slice_idx,game_idx,game_id
0,0,0,2362-28cbc35e-2542-495b-8053-8ab762f7dad9
1,0,1,9955-1cbd5506-e782-422f-84e7-70366d4f805c
2,0,2,3763-d036c1f6-aeab-4a7a-a55c-4bfe7ca1463a
3,0,3,6113-38555c00-9ed9-463a-ab14-04cdfb258539
4,0,4,7086-79b4b0a8-9d2c-4f84-920d-d788aaaf54d3
5,0,5,8112-d96bdbf4-aec9-4b5a-812b-2a525055ebe4
6,0,6,4011-91ffe3c6-22fb-46a6-9455-4216bc23a30a
7,0,7,1906-40c4b2ef-171b-4f0b-b70e-fb0955631406
8,0,8,6742-30dc1dc7-31a4-4f0d-9648-7e93930249e1
9,0,9,9597-b2286185-8375-4c3a-81a1-842be70a921e
